In [0]:
# cut ingestion_timestamp into ingestion_date and ingestion_time
# cứ để nó là string trước, xong rồi khi read từ bronze table, extract thành timestamp để lưu lại trong silver layer

## Transform from `bronze.playback` table
Transformation includes:
- Remove duplicates (because each pull from api gets 50 recent played tracks -> might overlap if no new incoming data)
- Cast `played_at` and `ingestion_timestamp` to be **timestamp**
- Separate `played_at` to be `played_date` and `played_time`
- Add a new column `updated_at` (i.e., now) to show the time when data is processed/updated into `silver` table

Finally, write into `silver` table


In [0]:
%sql
SELECT (*)
FROM spotify_dev.`01_bronze`.playback

#### Remove duplicates && Cast timestamp type

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_transformed_playback AS 
(  
  SELECT track_id, album_id, file_source,
          try_to_timestamp(played_at, 'dd-MM-yyyy HH:mm:ss') AS played_at_timestamp,  
          try_to_timestamp(ingestion_timestamp, 'dd-MM-yyyy HH:mm:ss') AS ingestion_timestamp 
  FROM 
  (
    SELECT *, 
            ROW_NUMBER() OVER (
              PARTITION BY track_id, album_id, played_at
              ORDER BY played_at DESC
            ) AS rn
    FROM spotify_dev.`01_bronze`.playback
  )
  WHERE rn = 1
)

In [0]:
%sql
SELECT *
FROM v_transformed_playback

## 

In [0]:
%sql
CREATE OR REPLACE TABLE spotify_dev.`02_silver`.dim_playback AS
(
  SELECT track_id, album_id, file_source,
    date_format(played_at_timestamp, 'dd-MM-yyyy') AS played_date, -- convert to timestamp format
    date_format(played_at_timestamp, 'HH:mm:ss') AS played_time,
    ingestion_timestamp,
    try_to_timestamp(
      date_format(from_utc_timestamp(current_timestamp(), "Europe/Amsterdam"), "dd-MM-yyyy HH:mm:ss"), 
      "dd-MM-yyyy HH:mm:ss"
    ) AS updated_at
  FROM v_transformed_playback
)

In [0]:
%sql
-- verify again if there is still duplicate
SELECT COUNT(*)
FROM 
(
  SELECT track_id, album_id, played_date, played_time, COUNT(*)
  FROM spotify_dev.`02_silver`.dim_playback
  GROUP BY track_id, album_id, played_date, played_time
  HAVING COUNT(*) > 1
  ORDER BY COUNT(*) DESC
)


## Transform from `bronze.artist` table
Transformation includes:
- Remove duplicates (same artist can appear as in different tracks)
- Add a new column `updated_at` (i.e., now) to show the time when data is processed/updated into `silver` table

Finally, write into `silver` table


In [0]:
%sql 
-- original from bronze table
SELECT *
FROM spotify_dev.`01_bronze`.artist

#### Deduplicate and Create a temporary view to save the unique artists

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_unique_artist AS
(
    SELECT * 
    FROM 
    ( -- table with row number and id being a partition 
        SELECT *, 
                ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY name ASC
                ) AS rn
        FROM spotify_dev.`01_bronze`.artist
    )
    WHERE rn = 1
)


In [0]:
%sql
-- write into silver table
CREATE OR REPLACE TABLE spotify_dev.`02_silver`.dim_artist AS
(
    SELECT id, name, file_source, ingestion_timestamp, array_distinct(from_json(genres, 'ARRAY<STRING>')) AS genres_array,
    try_to_timestamp(
      date_format(from_utc_timestamp(current_timestamp(), "Europe/Amsterdam"), "dd-MM-yyyy HH:mm:ss"), 
      "dd-MM-yyyy HH:mm:ss"
    ) AS updated_at
    FROM v_unique_artist 
)


In [0]:
%sql
-- verify again if there is still duplicate
SELECT COUNT(*)
FROM 
(
  SELECT id, name, COUNT(*)
  FROM spotify_dev.`02_silver`.dim_artist
  GROUP BY id, name
  HAVING COUNT(*) > 1
  ORDER BY COUNT(*) DESC
)


In [0]:
%sql
SELECT *
FROM spotify_dev.`02_silver`.dim_artist

## Transform from `bronze.album` table
Transformation includes:
- Remove duplicates (an album can appear as many tracks belong to 1 album)
- Change data type of column `release_date` to `timestamp`
- Normalize column `release_date` since it contains mixed values (e.g., _2007, 1957-09, 2002-08-27_)
- Add a new column `updated_at` (i.e., now) to show the time when data is processed/updated into `silver` table

Finally, write into `silver` table


In [0]:
%sql 
-- original from bronze table
SELECT *
FROM spotify_dev.`01_bronze`.album


#### Deduplicate and Create a temporary view to save the unique albums

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_unique_album AS
(
    SELECT  id, name, album_type, 
    CASE
        WHEN release_date RLIKE '^[0-9]{4}$' -- only year
          THEN CONCAT(release_date, '-01-01')
        WHEN release_date RLIKE '^[0-9]{4}-[0-9]{2}$' -- only year and month
          THEN CONCAT(release_date, '-01')
        WHEN release_date RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}$' -- full form year, month and day
          THEN release_date
        ELSE NULL
      END
    AS release_date, 
    ingestion_timestamp, total_tracks, file_source
    FROM 
    ( -- table with row number and id being a partition 
        SELECT *, 
                ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY name ASC
                ) AS rn
        FROM spotify_dev.`01_bronze`.album
    )
    WHERE rn = 1
)


In [0]:
%sql
select * from v_unique_album order by release_date

In [0]:
%sql
-- write into silver table
CREATE OR REPLACE TABLE spotify_dev.`02_silver`.dim_album AS
(
    SELECT id, name, album_type,
    to_timestamp(release_date, 'yyyy-MM-dd') AS release_date,
    total_tracks, file_source, ingestion_timestamp,
    try_to_timestamp(
      date_format(from_utc_timestamp(current_timestamp(), "Europe/Amsterdam"), "dd-MM-yyyy HH:mm:ss"), 
      "dd-MM-yyyy HH:mm:ss"
    ) AS updated_at
    FROM v_unique_album 
)


In [0]:
%sql
select * from spotify_dev.`02_silver`.dim_album

## Transform from `bronze.track` table
Transformation includes:
- Remove duplicates (tracks can be duplicated due to overlap)
- Change data type of `album` column from string -> json object
- Change data type of `artists` column from string -> array of json objects 
- Add a new column `updated_at` (i.e., now) to show the time when data is processed/updated into `silver` table

Finally, write into `silver` table


In [0]:
%sql 
-- original from bronze table
SELECT *
FROM spotify_dev.`01_bronze`.track
LIMIT 50


#### Deduplicate and Create a temporary view to save the unique tracks

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_unique_track AS
(
    SELECT * 
    FROM 
    ( -- table with row number and id being a partition 
        SELECT *, 
                ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY name ASC
                ) AS rn
        FROM spotify_dev.`01_bronze`.track
    )
    WHERE rn = 1
)


In [0]:
%sql
select * from v_unique_track

In [0]:
%sql
select schema_of_json(artists) 
from v_unique_track
limit 1

In [0]:
%sql
-- write into silver table
CREATE OR REPLACE TABLE spotify_dev.`02_silver`.dim_track AS
(
    SELECT id, name, 
    album:id AS album_id,
    album:name AS album_name, 
    album:album_type AS album_type,
    album:release_date::TIMESTAMP AS album_release_date,
    album:total_tracks:: INTEGER AS album_total_tracks,
    array_distinct(from_json(artists, "ARRAY<STRUCT<
        id: STRING,
        name: STRING,
        genres: ARRAY<STRING>
    >>")) AS artists_array, 
    popularity, duration_ms, file_source, ingestion_timestamp,
    try_to_timestamp(
      date_format(from_utc_timestamp(current_timestamp(), "Europe/Amsterdam"), "dd-MM-yyyy HH:mm:ss"), 
      "dd-MM-yyyy HH:mm:ss"
    ) AS updated_at
    FROM v_unique_track 
)


In [0]:
%sql
select * from spotify_dev.`02_silver`.dim_track